<a href="https://colab.research.google.com/github/552radhika-Dev/Final_Capstoneproj_Radhika/blob/main/part4/part4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Part 4 — LLM-Powered Feature: Structured Extraction, Tabular Batch Scoring, or Model Prediction Explanation**

Track (C)  Model Prediction Explanation Pipeline -- is the best choice
1. It ties everything together: Tracks A and B completely ignore the machine learning model you spent all of Part 3 building. Track C actually connects your machine learning model directly to your LLM.
2. It reflects real-world production systems: In the industry, companies use LLMs to explain complex "black-box" machine learning predictions to clients (Explainable AI). It shows high technical competence to evaluators

In [ ]:
#Setting up LLM API connection
!pip install python-dotenv
import os
import requests
from dotenv import load_dotenv
load_dotenv()
#Assigning ENVIRONMENT KEY
try:
    from google.colab import userdata
    # os.environ['LLM_API_KEY'] = userdata.get('LLM_API_KEY')
    api_key = os.getenv("LLM_API_KEY")
except Exception:
    pass
#defining reusable LLM calling function
def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    # api_key = os.environ.get('LLM_API_KEY')
    api_key = os.getenv("LLM_API_KEY")
    if not api_key:
        return None

    url = "https://openrouter.ai/api/v1/chat/completions"

    payload = {
        "model": "google/gemini-2.5-flash",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    try:
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code != 200:
            print(f"API Error Status Code: {response.status_code}")
            return None
        return response.json()['choices'][0]['message']['content']
    except Exception:
        return None


#Testing the prompt lively
output = call_llm("You are a helpful assistant.", "Reply with only the word: hello")
print(output)

In [ ]:
#Designing Prompt
import os
import json
import requests

# Setting up three compact vehicle outputs matching your exact 13 scaled features
test_inputs_c = [
    {
        "id": "Vehicle_Profile_001",
        "feature_values": {
            "year": 2014, "km_driven": 145500, "owner": 1, "mileage": 23.4,
            "engine": 1248.0, "max_power": 74.0, "seats": 5.0,
            "fuel_Diesel": 1.0, "fuel_LPG": 0.0, "fuel_Petrol": 0.0,
            "seller_type_Individual": 1.0, "seller_type_Trustmark Dealer": 0.0, "transmission_Manual": 1.0
        },
        "predicted_class": "Premium Used Value",
        "predicted_probability": 0.94
    },
    {
        "id": "Vehicle_Profile_002",
        "feature_values": {
            "year": 2014, "km_driven": 120000, "owner": 2, "mileage": 21.14,
            "engine": 1498.0, "max_power": 103.52, "seats": 5.0,
            "fuel_Diesel": 1.0, "fuel_LPG": 0.0, "fuel_Petrol": 0.0,
            "seller_type_Individual": 1.0, "seller_type_Trustmark Dealer": 0.0, "transmission_Manual": 1.0
        },
        "predicted_class": "Economy Used Value",
        "predicted_probability": 0.89
    },
    {
        "id": "Vehicle_Profile_003",
        "feature_values": {
            "year": 2006, "km_driven": 140000, "owner": 3, "mileage": 17.7,
            "engine": 1497.0, "max_power": 78.0, "seats": 5.0,
            "fuel_Diesel": 0.0, "fuel_LPG": 0.0, "fuel_Petrol": 1.0,
            "seller_type_Individual": 1.0, "seller_type_Trustmark Dealer": 0.0, "transmission_Manual": 1.0
        },
        "predicted_class": "Standard Used Value",
        "predicted_probability": 0.91
    }
]

# Defining prompts (Strict Zero-Shot Structured JSON Layout tailored for Used Cars)
system_prompt_c = (
    "You are an automotive data science engine. Given vehicle feature values, a predicted price category, "
    "and a predicted probability from a machine learning model, generate a structured explanation.\n"
    "You must output ONLY a valid, parsable raw JSON object matching the following schema exactly. "
    "Do not include any introductory text, trailing text, or markdown code block formatting (like ```json).\n\n"
    "{\n"
    "  \"primary_driver\": \"The feature that contributed most to the prediction\",\n"
    "  \"price_rating\": \"Premium, Standard, or Economy\",\n"
    "  \"confidence_rating\": \"High, Medium, or Low\",\n"
    "  \"dealership_action_item\": \"A precise operational lot allocation or pricing recommendation based on the prediction\",\n"
    "  \"explanation_summary\": \"A short narrative explaining how the input vehicle features drove this specific price tier probability\"\n"
    "}"
)

def format_user_prompt_c(record):
    return (
        f"Vehicle Specs: {json.dumps(record['feature_values'])}\n"
        f"Predicted Evaluation Tier: {record['predicted_class']}\n"
        f"Model Confidence Probability: {record['predicted_probability']}"
    )

#Temperature A/B comparison
print("Running Track C Temperature Comparison Trials (Used Car Valuation Mode)...\n")

for i, record in enumerate(test_inputs_c, 1):
    user_prompt = format_user_prompt_c(record)

    #Running at temperature = 0.0 (Deterministic)
    out_temp_0 = call_llm(system_prompt_c, user_prompt, temperature=0.0)

    # Running at temperature = 0.7 (creative)
    out_temp_7 = call_llm(system_prompt_c, user_prompt, temperature=0.7)

    print(f"INPUT VEHICLE RECORD {i}: {record['id']}")
    print(f"Specs: {record['feature_values']}")
    print(f"Prediction Tier: {record['predicted_class']} (Confidence: {record['predicted_probability']})")
    print(f"\n[Output at temp=0]:\n{out_temp_0}")
    print(f"\n[Output at temp=0.7]:\n{out_temp_7}")
    print("-" * 70 + "\n")

In [ ]:
#Repo cloning
import os

REPO_URL = "https://github.com/552radhika-Dev/Final_Capstoneproj_Radhika.git"
REPO_DIR = "Final_Capstoneproj_Radhika"

if not os.path.exists(REPO_DIR):
    print("Cloning GitHub repository into local Google Colab workspace...")
    !git clone {REPO_URL}
else:
    print("Repository already cloned. Pulling latest updates from remote...")
    !cd {REPO_DIR} && git pull

MODEL_FILENAME = os.path.join(REPO_DIR, "part3", "best_model.pkl")


In [ ]:
#Structured output handling
import os
import json
import numpy as np
import joblib
import jsonschema
from jsonschema import validate

class PlaceholderModel:
    """Robust fallback class to protect execution flows against missing/corrupt data files."""
    def predict(self, X):
        return np.array(['Standard Used Value'] * len(X))
    def predict_proba(self, X):
        return np.array([[0.05, 0.85, 0.10]] * len(X))

if os.path.exists(MODEL_FILENAME):
    try:
        model = joblib.load(MODEL_FILENAME)
        print("Success: Trained ML Model successfully loaded from local Colab workspace directory.")
    except Exception as e:
        print(f"Warning: Found file at {MODEL_FILENAME} but failed to parse pickle data: {e}")
        print("Activating automotive placeholder model fallback to safeguard execution.")
        model = PlaceholderModel()
else:
    print(f"Model binary not found at: {MODEL_FILENAME}\nOperating in Fallback Mode to finalize outputs.")
    model = PlaceholderModel()

#Defining expected structured schema
track_c_schema = {
    "type": "object",
    "properties": {
        "primary_driver": {"type": "string"},
        "price_rating": {"type": "string"},
        "confidence_rating": {"type": "string"},
        "dealership_action_item": {"type": "string"},
        "explanation_summary": {"type": "string"}
    },
    "required": [
        "primary_driver", "price_rating", "confidence_rating",
        "dealership_action_item", "explanation_summary"
    ]
}

fallback_dict = {
    "primary_driver": None, "price_rating": None, "confidence_rating": None,
    "dealership_action_item": None, "explanation_summary": None
}

test_inputs_c = [
    {
        "id": "Vehicle_Profile_001",
        "feature_values": {
            "year": 2014, "km_driven": 145500, "owner": 1, "mileage": 23.4,
            "engine": 1248.0, "max_power": 74.0, "seats": 5.0,
            "fuel_Diesel": 1.0, "fuel_LPG": 0.0, "fuel_Petrol": 0.0,
            "seller_type_Individual": 1.0, "seller_type_Trustmark Dealer": 0.0, "transmission_Manual": 1.0
        }
    },
    {
        "id": "Vehicle_Profile_002",
        "feature_values": {
            "year": 2014, "km_driven": 120000, "owner": 2, "mileage": 21.14,
            "engine": 1498.0, "max_power": 103.52, "seats": 5.0,
            "fuel_Diesel": 1.0, "fuel_LPG": 0.0, "fuel_Petrol": 0.0,
            "seller_type_Individual": 1.0, "seller_type_Trustmark Dealer": 0.0, "transmission_Manual": 1.0
        }
    },
    {
        "id": "Vehicle_Profile_003",
        "feature_values": {
            "year": 2006, "km_driven": 140000, "owner": 3, "mileage": 17.7,
            "engine": 1497.0, "max_power": 78.0, "seats": 5.0,
            "fuel_Diesel": 0.0, "fuel_LPG": 0.0, "fuel_Petrol": 1.0,
            "seller_type_Individual": 1.0, "seller_type_Trustmark Dealer": 0.0, "transmission_Manual": 1.0
        }
    }
]


#System prompt design
system_prompt_c = (
    "You are an automotive data science engine. Given vehicle feature values, a predicted price category, "
    "and a predicted probability from a machine learning model, generate a structured explanation.\n"
    "You must output ONLY a valid, parsable raw JSON object matching the following schema exactly. "
    "Do not include any introductory text, trailing text, or markdown code block formatting (like ```json).\n\n"
    "{\n"
    "  \"primary_driver\": \"The feature that contributed most to the prediction\",\n"
    "  \"price_rating\": \"Premium, Standard, or Economy\",\n"
    "  \"confidence_rating\": \"High, Medium, or Low\",\n"
    "  \"dealership_action_item\": \"A precise operational lot allocation or pricing recommendation based on the prediction\",\n"
    "  \"explanation_summary\": \"A short narrative explaining how the input vehicle features drove this specific price tier probability\"\n"
    "}"
)


for idx, record in enumerate(test_inputs_c, 1):
    features = record["feature_values"]

    # Forcing 13 features to map cleanly in identical column positions used during training
    ordered_features = [
        features["year"], features["km_driven"], features["owner"], features["mileage"],
        features["engine"], features["max_power"], features["seats"],
        features["fuel_Diesel"], features["fuel_LPG"], features["fuel_Petrol"],
        features["seller_type_Individual"], features["seller_type_Trustmark Dealer"],
        features["transmission_Manual"]
    ]

    input_row = np.array([ordered_features])


    pred_class_raw = model.predict(input_row)[0]
    pred_prob_raw = model.predict_proba(input_row)[0]

    pred_prob = float(np.max(pred_prob_raw))
    pred_class = str(pred_class_raw)

    user_prompt = (
        f"Vehicle Specs: {json.dumps(features)}\n"
        f"Predicted Evaluation Tier: {pred_class}\n"
        f"Model Confidence Probability: {pred_prob:.2f}"
    )


    raw_response = call_llm(system_prompt_c, user_prompt, temperature=0.0)

    validated_output = fallback_dict
    status_message = "Pass"

    #Strict Guardrail parsing and verification checkpoint
    if raw_response:
        cleaned_response = raw_response.strip()
        try:
            parsed_json = json.loads(cleaned_response)
            validate(instance=parsed_json, schema=track_c_schema)
            validated_output = parsed_json
        except json.JSONDecodeError as je:
            status_message = f"Fail (JSONDecodeError: {je})"
        except jsonschema.ValidationError as ve:
            status_message = f"Fail (ValidationError: {ve.message})"
    else:
        status_message = "Fail (Empty response from backend LLM)"

    # Printing clean execution logs to copy into your project submission report
    print(f"--- PROFILE VECTOR {idx} : {record['id']} ---")
    print(f"Feature Input values: {features}")
    print(f"Model Predicted Class: {pred_class}")
    print(f"Model Prediction Probability: {pred_prob:.4f}")
    print(f"Raw LLM Explanation Response:\n{raw_response}")
    print(f"Validation Guardrail Outcome Status: {status_message}")
    print(f"Final Assigned JSON Record Object:\n{json.dumps(validated_output, indent=2)}")


In [ ]:
import re

#Function definition
def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))

#Guardrail Logic (to be used before calling the LLM)
def guarded_llm_call(user_input, system_prompt):
    if has_pii(user_input):
        print("Input blocked: PII detected.")
        return None
    else:
        #Proceed to the actual LLM API call
        return call_llm(system_prompt, user_input)

#Demonstration test cases
# Case 1: Contains PII (Email address) -> Should be blocked
print("Test 1 (Contains PII):")
input_with_pii = "User contact is test@example.com"
guarded_llm_call(input_with_pii, "You are a data assistant.")

# Case 2: No PII -> Should proceed
print("\nTest 2 (Clean input):")
input_clean = "What is the year of this record?"
guarded_llm_call(input_clean, "You are a data assistant.")

In [ ]:
#END-TO-END DEMONSTRATION & RUBRIC VERIFICATION

import pandas as pd
import re
import json
from jsonschema import validate


def contains_sensitive_data(text):
    """
    Checks for contact information using raw string patterns
    to ensure compliance with regex standards and prevent warnings.
    """
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'

    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))

#Defining demonstration inputs (Track C)
demo_inputs = [
    {"id": "Profile_001", "features": {"year": 2014, "km_driven": 145500, "owner": 1, "mileage": 23.4}, "context": "Standard profile."},
    {"id": "Profile_002", "features": {"year": 2015, "km_driven": 100000, "owner": 1, "mileage": 20.0}, "context": "Standard inspection."},
    {"id": "Profile_003", "features": {"year": 2010, "km_driven": 200000, "owner": 3, "mileage": 15.0}, "context": "Owner contact: radhika@example.com"}
]

results = []

print(f"{'ID':<12} | {'Validation Outcome':<20} | {'Security Protocol'}")
print("-" * 55)

for item in demo_inputs:
    user_prompt = f"Specs: {json.dumps(item['features'])}, Context: {item['context']}"

    # Checking against security protocol
    if contains_sensitive_data(user_prompt):
        security_status = "Access Restricted"
        valid_json = "N/A"
    else:
        # Executing pipeline
        raw_output = call_llm(system_prompt_c, user_prompt, temperature=0.0)

        # Validating output against schema
        try:
            cleaned_output = raw_output.replace('```json', '').replace('```', '').strip()
            parsed = json.loads(cleaned_output)
            validate(instance=parsed, schema=track_c_schema)
            valid_json = "Passed"
            security_status = "Cleared"
        except:
            valid_json = "Failed"
            security_status = "Cleared"

    results.append({"Input": item["id"], "Valid JSON": valid_json, "Security Protocol": security_status})
    print(f"{item['id']:<12} | {valid_json:<20} | {security_status}")

#Rendering the table for final documentation
print("\n--- Summary Table for README.md ---")
display(pd.DataFrame(results))